# 第11回：化学の知識を特徴量にする

**今日の問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：仮説を計算式にする

最適温度78℃からの距離、単位時間あたりの濃度という2つの仮説特徴量を作ります。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = abs(engineered["temperature_c"] - 78)
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


## 同じ検証条件で追加前後を比べる


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
train_idx, valid_idx = train_test_split(engineered.index, test_size=0.25, random_state=42)
for name, columns in {"追加前": base, "追加後": added}.items():
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42))
    model.fit(engineered.loc[train_idx, columns], engineered.loc[train_idx, "yield_pct"])
    pred = model.predict(engineered.loc[valid_idx, columns])
    print(name, "MAE:", round(mean_absolute_error(engineered.loc[valid_idx, "yield_pct"], pred), 3))


## CHALLENGE：RDKitでSMILESから記述子を再計算


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 3))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 3))
except ImportError:
    print("RDKitは任意です。計算済みのmolecular_weight、logp、tpsa列で本編を進められます。")


## CHANGE

自分の化学的仮説を1つ選び、計算式・予測時点・期待する方向を先に書いてから列を作ります。改善しなくても有益な結果です。
